# Multi-Agent Agent Valley Training

This notebook demonstrates the Theme #1 multi-agent layer. It uses small CPU-safe PyTorch policies, not full transformer-scale TRL/Unsloth training.

Optional setup in Colab:

```python
# !pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'env').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## Reset And Step All Four Agents

In [ ]:
from env.multi_agent_env import MultiAgentValleyEnv

env = MultiAgentValleyEnv(difficulty='hard', seed=42)
obs = env.reset()
actions = {
    'farmer':  {'primary_action': 'gather', 'focus_resource': 'food',  'cooperation_mode': 'share',      'risk_posture': 'balanced', 'rationale': 'Food is low, so I will feed the team.'},
    'miner':   {'primary_action': 'gather', 'focus_resource': 'ore',   'cooperation_mode': 'coordinate', 'risk_posture': 'balanced', 'rationale': 'Ore supports coordinated construction.'},
    'builder': {'primary_action': 'build',  'focus_resource': 'stone', 'cooperation_mode': 'coordinate', 'risk_posture': 'cautious',  'rationale': 'Stone building improves defenses.'},
    'warrior': {'primary_action': 'defend', 'focus_resource': 'none',  'cooperation_mode': 'protect',    'risk_posture': 'cautious',  'rationale': 'Threat is high, so I will protect the team.'},
}
next_obs, rewards, dones, info = env.step(actions)
print('agents:', sorted(obs))
print('rewards:', rewards)
print('dones:', dones)
print('lead agent:', info['lead_agent'])
print('cooperation bonus:', info['cooperation_bonus'])

## Partial Observations

In [ ]:
for agent_id, agent_obs in obs.items():
    visible = {k: agent_obs[k] for k in ['agent_id', 'agent_role', 'agent_home_region', 'region', 'food_supply', 'ore_supply', 'threat_level']}
    print(agent_id, visible)

## Reward Breakdown

In [ ]:
json.dumps(info['reward_components_by_agent'], indent=2)

## Tiny Multi-Agent GRPO Run

In [ ]:
from training.ma_grpo_train import MAGRPOConfig, MAGRPOTrainer

trainer = MAGRPOTrainer(MAGRPOConfig(episodes=5, difficulty='easy', seed=42, group_size=3, reset_metrics=True))
metrics = trainer.train()
metrics[-1]

## Plot Team Reward And Per-Agent Role Rewards

In [ ]:
try:
    import matplotlib.pyplot as plt
    episodes = [row['episode'] for row in metrics]
    plt.figure(figsize=(8, 3))
    plt.plot(episodes, [row['total_team_reward'] for row in metrics], marker='o')
    plt.title('Multi-agent team reward')
    plt.xlabel('Episode')
    plt.ylabel('Team reward')
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 3))
    for agent_id in metrics[-1]['agent_metrics']:
        plt.plot(episodes, [row['agent_metrics'][agent_id]['role_reward'] for row in metrics], marker='o', label=agent_id)
    plt.title('Per-agent role reward')
    plt.xlabel('Episode')
    plt.ylabel('Role reward')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
except ImportError:
    print('matplotlib is optional; inspect metrics directly:', metrics[-1])

## Untrained vs Trained Action Example

In [ ]:
from training.neural_policy import select_action

fresh = MAGRPOTrainer(MAGRPOConfig(episodes=1, difficulty='easy', seed=99, group_size=2, reset_metrics=False))
env = MultiAgentValleyEnv(difficulty='easy', seed=42)
obs = env.reset()
_, untrained_action = select_action(fresh.policies['farmer'], obs['farmer'], deterministic=True)
_, trained_action = select_action(trainer.policies['farmer'], obs['farmer'], deterministic=True)
print('untrained farmer:', untrained_action)
print('trained farmer:', trained_action)